In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score, mean_squared_error




In [ ]:
# Cargar el dataset
df = pd.read_csv('ventas_ml_clase2.csv')

df.head()

In [ ]:
df.describe(include='all').transpose().head()

In [ ]:
#predecir las ventas usando las variables: marketing, precio, temporada, tienda, canal
x = df[['marketing', 'precio', 'temporada', 'tienda', 'canal']]
y = df['ventas']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {x_train.shape[0]} filas ({(x_train.shape[0]/len(df))*100:.0f}%) del total")
print(f"Datos de prueba: {x_test.shape[0]} filas ({(x_test.shape[0]/len(df))*100:.0f}%) del total")
print()
print("El modelo aprendera con el 80% de los datos y se evaluara con el 20% restante")
print("Esto evitara que el modelo memorice los datos y permita evaluar su capacidad de generalizacion a datos nuevos")


In [ ]:
numerical_features = ['marketing', 'precio', 'temporada']
categorical_features = ['tienda', 'canal']

preprocess = ColumnTransformer(
    transformers=(
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    )
)
model = LinearRegression()

pipe = Pipeline(steps=[
    ('preprocess', preprocess), 
    ('model', model)
])
pipe    

In [ ]:
pipe.fit(x_train, y_train)  
 #entrenar el modelo con los datos de entrenamiento

In [ ]:
#generar predicciones con los datos de prueba
y_pred = pipe.predict(x_test)
mae = mean_absolute_error(y_test, y_pred) #calcular el error absoluto medio
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) #calcular la raiz del error cuadratico medio
r2 = r2_score(y_test, y_pred) #r2 score proporcin de la variableexpñicada por el modelo 

media_ventas = y_test.mean() #calcular la media de las ventas reales

print(f"MAE (error absoluto medio): ${mae:.2f}")
print(f"RMSE (raiz del error cuadratico medio): ${rmse:.2f}")
print(f"R2 Score proporcion de varianza explicada {r2:.4f} ({r2*100:.1f}%)")

print("="*50)

print(f"Media de ventas reales: ${media_ventas:.2f}")
print("Error relativo (MAE/media:)", f"{(mae/media_ventas)*100:.1f}%")


print("="*50)

print("Interpretacion de las metricas:")
print(f"MAE: ${mae:.2f} - Error promedio entre predicciones y valores reales")
print(f"RMSE: ${rmse:.2f} - Raiz del error cuadratico medio")
print(f"R2 Score: {r2:.4f} - Proporcion de varianza explicada por el modelo")
print(f"Error relativo: {(mae/media_ventas)*100:.1f}%")



In [ ]:
# --- Gráfica de Ventas Reales vs Predichas con línea de tendencia ---
fig, ax = plt.subplots(figsize=(8, 6))

# Cada punto es una observación del set de prueba
# Eje X = venta real, Eje Y = venta predicha por el modelo
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Predicciones")

# Línea de predicción perfecta (donde real = predicho)
# Si el modelo fuera perfecto, todos los puntos estarían sobre esta línea
limite_min = min(y_test.min(), y_pred.min()) - 10
limite_max = max(y_test.max(), y_pred.max()) + 10
ax.plot([limite_min, limite_max], [limite_min, limite_max],
        color="red", linestyle="--", linewidth=1.5,
        label="Prediccion perfecta (real = predicho)")

# Etiquetas y título
ax.set_xlabel("Ventas reales ($)", fontsize=12)
ax.set_ylabel("Ventas predichas ($)", fontsize=12)
ax.set_title(f"Ventas reales vs predichas  (R² = {r2:.2f})", fontsize=14)
ax.legend(fontsize=10)

# Ajustar los ejes para que tengan la misma escala
ax.set_xlim(limite_min, limite_max)
ax.set_ylim(limite_min, limite_max)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# Coeficiente del modelo: ¿Que variables pesan mas?

Un modelo de regresion lineal es una forma de ***suma ponderada***

ventas = base +(peso x marqueting) + (Peso2 x precio ) + (peso3 x temporada) + ...

In [ ]:
# Exrtaer los coeficientes del modelo lineal para interpretar la importancia de cada variable

#obtener los nombres de las columnas categoricas despues del OneHotEncoding
ohe = pipe.named_steps['preprocess'].named_transformers_['cat']
cat_cols = ohe.get_feature_names_out({"tienda", "canal"}).tolist()

# Liara completa de variables: numericas + las generadas por el OneHotEncoding
feature_names = numerical_features + cat_cols

# Los coeficientes son los "pesos" de la formula
# ventas = intercepto + coef1*marketing + coef2*precio + ... + coefN*canal_online
# ventas = intercepto + coef1*marketing + coef2*precio + coef3*temporada + coef4*tienda_A + coef5*tienda_B + coef6*canal_online
coef = pipe.named_steps['model'].coef_ #coeficientes de cada variable

#organizar de mayor a menor
coef_df = pd.DataFrame({
    "variable": feature_names,
    "coef": coef
}).sort_values(by="coef", key=abs, ascending=False)


#mostrar la tabla de coeficientes
print("="*50)
print("Coeficientes del modelo ")
for _, fila in coef_df.iterrows():
    signo = "+" if coef > 0 else ""
    barra ="^" if coef > 0 else "v"